# 01 · 메인 학습 — aloha insertion (150k / seed 4개)

**프로토콜 (확정)**
- **150k step 까지만** 학습 (`cf.STEPS`). 그 다음 `02_eval_main` 이 **150k 체크포인트**를 평가.
- **seed 4개** (`cf.MAIN_SEEDS = [0,1,2,3]`) — 해준/은지가 나눠 돌리고 pooled.
- **lr sweep 없음.** 전 모델 고정 **1e-5** (diffusion/smolvla 만 원 논문 1e-4 — baseline 공정성).
- 학습 모델 = `cf.TRAIN_TAGS` 6개 (`act`·`diffusion`·`smolvla`·`acm2`·`acm`·`ours`).
  `act_te` 는 학습 안 함 → eval 때 act 체크포인트에 TE 를 얹음.

⚠️ 먼저 **parity 테스트**: 클러스터에서 `python tests/test_acm_sscp_literal.py`
(acm carry 정확성. 깨지면 `ours` 학습이 무의미).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM          # 'insertion'
SEEDS = cf.MAIN_SEEDS        # [0, 1, 2, 3]  (은지와 나눠 돌릴 땐 여기만 바꿈: 예 [0,1])
NGPU  = 8

cf.check_tags()
print()
print('task :', TASK, cf.v23.TASKS[TASK], '| fps', cf.fps_of(TASK))
print('steps:', f'{cf.STEPS:,}', '| seeds:', SEEDS, '| NGPU:', NGPU)
print('학습 태그:', cf.TRAIN_TAGS, '(act_te 제외 — eval-time TE)')
print('총 잡 수:', len(cf.TRAIN_TAGS) * len(SEEDS))

## 커맨드 확인 (dry-run) — lr / steps / 플래그

In [ ]:
for t in cf.TRAIN_TAGS:
    c = cf.make_train_cmd(t, seed=SEEDS[0], task=TASK, gpu_id=0)
    lr = [p for p in c.split() if p.startswith('--policy.optimizer_lr')][0]
    st = [p for p in c.split() if p.startswith('--steps')][0]
    print(f'{t:<10} {lr:<28} {st}')
print()
print(cf.make_train_cmd('ours', seed=SEEDS[0], task=TASK, gpu_id=0))

## 학습 — NGPU 만큼 청크로 (각 청크 끝날 때까지 대기, resume 자동)

In [ ]:
jobs = cf.train_jobs(SEEDS, task=TASK)          # (tag, seed, task) x 6모델 x seed
print('총', len(jobs), '잡')
for i in range(0, len(jobs), NGPU):
    chunk = jobs[i:i + NGPU]
    print('\n===== 청크 %d/%d (%d 잡) =====' % (i // NGPU + 1, -(-len(jobs) // NGPU), len(chunk)))
    for j in chunk:
        print('  ', j)
    cf.launch_training_live(chunk)
print('\n메인 학습 완료 (150k):', SEEDS)

## 상태 — 150k 도달 여부

In [ ]:
cf.print_training_status(jobs)
print()
print('150k 체크포인트 존재 확인:')
for s in SEEDS:
    row = []
    for t in cf.TRAIN_TAGS:
        cd = cf.v23.best_ckpt_dir(t, s, TASK, how=cf.CKPT_STEP)
        row.append(f"{t}:{'OK' if cd is not None and int(cd.name) == cf.CKPT_STEP else (cd.name if cd is not None else 'X')}")
    print(f'  seed{s}  ' + '  '.join(row))